# Task 1

### 1. Diferencia Fundamental: Defina con sus propias palabras la diferencia exacta a nivel de píxel entre la Segmentación Semántica y la Segmentación de Instancia.

# 1. Diferencia Fundamental: Segmentación Semántica vs Segmentación de Instancia

## Definición general

La diferencia entre segmentación semántica y segmentación de instancia está en cómo se etiquetan los píxeles dentro de una imagen.

---

## Segmentación semántica

En la segmentación semántica, cada píxel se clasifica únicamente según su clase, sin importar a qué objeto específico pertenece.

Por ejemplo:
- Si hay varias personas en la imagen, todos los píxeles correspondientes a personas se etiquetan simplemente como "persona".
- Si hay varias espadas, todas se etiquetan como "espada", sin diferenciarlas entre sí.

Esto implica que:
- No hay distinción entre objetos individuales.
- Se obtiene una sola máscara por cada clase.

---

## Segmentación de instancia

En la segmentación de instancia, cada píxel no solo se clasifica por su clase, sino también por la instancia específica del objeto al que pertenece.

Por ejemplo:
- Persona 1 y Persona 2 se consideran objetos distintos.
- Espada 1 y Espada 2 también se separan, aunque sean iguales.

Esto permite:
- Diferenciar objetos individuales dentro de la misma clase.
- Generar una máscara independiente para cada objeto.

---

### Segmentación semántica

Cada píxel tiene solo una etiqueta de clase:

$$
pixel_i = clase
$$


### Segmentación de instancia

Cada píxel tiene una etiqueta de clase y una de instancia:

$$
pixel_i = (clase, instancia)
$$

---


## Referencias

1. He, K., Gkioxari, G., Dollár, P., & Girshick, R. (2017). Mask R-CNN. ICCV.  
   https://arxiv.org/abs/1703.06870


### 2. El Caso U-Net: Explique por qué la arquitectura U-Net (Segmentación Semántica) es ideal y altamente eficiente para la tarea 1 (separar a todos los humanos del fondo), pero explique técnicamente por qué fracasaría si intentamos usarla para la tarea 2 si hay dos réplicas de espadas idénticas cruzadas en la foto.


U-Net es una arquitectura diseñada específicamente para segmentación semántica, y funciona muy bien para separar clases generales como "humano" vs "fondo".

por su arquitectura: 
- El **encoder** extrae características de alto nivel, lo que ayuda a entender el contexto de la imagen (por ejemplo, qué partes corresponden a un humano).
- El **decoder** reconstruye la imagen a nivel de píxel, permitiendo obtener una máscara detallada.
- Las **skip connections** conectan encoder y decoder, lo que permite recuperar información espacial fina como bordes y contornos.

Gracias a esto, U-Net logra:
- Segmentaciones muy precisas a nivel de píxel
- Buena definición de siluetas humanas
- Buen desempeño incluso con fondos complejos

aprende como: 

$$
pixel_i = clase
$$

donde cada píxel se clasifica como "humano" o "fondo".

---


El problema aparece cuando se necesita distinguir objetos individuales de la **misma clase**.

Como es segmentación semántica:
- No distingue entre instancias
- Genera una sola **máscara** por clase

Es decir, todas las espadas en la imagen se etiquetan simplemente como "espada", sin diferenciar cuál es cuál:

$$
pixel_i = clase
$$

Si hay dos espadas idénticas cruzadas:
- El modelo no tiene un mecanismo para separarlas
- Las interpreta como una sola región continua si están en contacto o muy cerca

porque: 
- No existe una asignación de identidad de objeto (instancia)
- No hay separación basada en objetos, solo en clases


---

## Referencias

Hafiz, A. M., & Bhat, G. M. (2020).  
   A Survey on Instance Segmentation: State of the Art.  
   https://ieeexplore.ieee.org/document/9122429

Minaee, S., Boykov, Y., Porikli, F., Plaza, A., Kehtarnavaz, N., & Terzopoulos, D. (2021).  
   Image Segmentation Using Deep Learning: A Survey.  
   https://doi.org/10.1109/TPAMI.2021.3059968
   
---

### 3. El Caso Mask R-CNN: Explique cómo la arquitectura Mask R-CNN (Segmentación de Instancia)resuelve el problema anterior basándose en su naturaleza de "dos etapas" (recordando que hereda de Faster R-CNN). ¿Por qué Mask R-CNN sí podría iluminar una espada de rojo y otra de azul, incluso si se están tocando en la imagen? 



## Etapa 1

En la primera etapa, el modelo utiliza un **Region Proposal Network (RPN)** para generar regiones candidatas (bounding boxes) donde podrían existir objetos.

Luego, estas regiones se refinan y se clasifican (por ejemplo, "espada", "persona", etc.).

Con el conjunto de bounding boxes, cada uno asociado a un objeto específico.

---

## Etapa 2

En la segunda etapa, cada bounding box se procesa de forma independiente mediante una operación llamada **ROI Align**, que extrae una representación precisa de la región.

Cada objeto detectado tiene su propia máscara.

---

## ¿Por qué funciona incluso si las espadas se están tocando?

La clave está en que la detección ocurre antes de la segmentación:

- Cada espada es detectada como un objeto independiente en la etapa de bounding boxes
- Luego, cada región se segmenta por separado
- Las máscaras se generan de forma independiente, incluso si los objetos están en contacto

A nivel de píxel, esto se puede expresar como:

$$
pixel_i = (clase, instancia)
$$

Entonces dos objetos de la misma clase tengan etiquetas distintas.

---

## Aplicación al problema

Gracias a esta separación por instancia:

- Espada 1 tiene su propia máscara
- Espada 2 tiene otra máscara independiente

Esto permite aplicar efectos diferentes a cada una, por ejemplo:
- una espada con brillo rojo
- otra con brillo azul

incluso si son idénticas o están cruzadas en la imagen.

---


## Referencias

Ultralytics (2025).  
   What is Mask R-CNN and how does it work?  
   https://www.ultralytics.com/blog/what-is-mask-r-cnn-and-how-does-it-work

---

## Task 2

### 1. ¿Por qué el NMS hace que varios clones desaparezcan?

El problema no ocurre porque el modelo no vea a las personas, sino porque **las elimina después** durante la etapa de *Non-Maximum Suppression* (NMS). En una escena como la del concurso de cosplay, donde los 15 clones están abrazados, superpuestos y parcialmente ocluidos, muchas cajas predichas para personas distintas quedan muy cerca entre sí y comparten una gran parte de su área.

Esto se entiende con la fórmula del **IoU** (*Intersection over Union*):

$$
IoU(B_1, B_2) = \frac{|B_1 \cap B_2|}{|B_1 \cup B_2|}
$$

Donde:

- $B_1 \cap B_2$ representa el área de intersección entre dos cajas.
- $B_1 \cup B_2$ representa el área total cubierta por ambas cajas.

Si dos personas están muy juntas, las cajas que las rodean pueden tener un IoU alto aunque correspondan a **individuos diferentes**. El NMS conserva la caja con mayor confianza y elimina las demás si su IoU supera el umbral definido. Por eso, aunque la red haya generado muchas detecciones inicialmente, al final solo permanecen unas pocas en pantalla.

En este caso, el NMS interpreta incorrectamente que varias cajas pertenecen al mismo objeto, cuando en realidad representan a distintos clones muy cercanos. El resultado es la aparición de **falsos negativos**, porque varias personas reales desaparecen de la salida final.

---

### 2. ¿Qué pasaría si el umbral de IoU del NMS se ajusta a 0.15 o a 0.95?

Si el umbral de IoU se ajusta a **0.15**, el NMS se vuelve **muy estricto**. Eso significa que basta con que dos cajas se traslapen un poco para que una de ellas sea eliminada. En una escena de alta densidad como esta, donde casi todas las personas están pegadas unas a otras, muchas cajas correctas superarían fácilmente ese valor. Como consecuencia, el sistema descartaría demasiadas detecciones válidas y desaparecerían aún más personas reales. En otras palabras, un umbral tan bajo **empeora el problema de falsos negativos**.

Si el umbral de IoU se ajusta a **0.95**, el NMS se vuelve **muy permisivo**. En ese escenario, una caja solo se elimina si prácticamente coincide por completo con otra. Esto ayuda a conservar más detecciones de personas distintas, incluso si están muy cerca o parcialmente superpuestas. Sin embargo, también aumenta la probabilidad de que queden varias cajas repetidas sobre una misma persona, porque el sistema ya no filtraría casi ningún duplicado. El resultado sería una salida con más personas visibles, pero también con más **falsos positivos**.

Para este problema de **alta densidad y oclusión**, entre ambos valores el más adecuado sería **0.95**, porque el error más grave en esta escena no es que sobren cajas, sino que desaparezcan personas reales. En este tipo de situación conviene ser más permisivo para no eliminar detecciones correctas. Aun así, en una aplicación real normalmente se buscaría un valor intermedio-alto para equilibrar mejor la conservación de personas y la reducción de duplicados.

---

### 3. ¿Cómo resolvería YOLOv10 este problema con Dual Label Assignment sin tocar el NMS?

Si se pudiera cambiar el modelo a **YOLOv10**, el problema mejoraría desde la forma en que el modelo aprende a detectar objetos durante el entrenamiento. YOLOv10 introduce una estrategia llamada **Dual Label Assignment**, que combina una asignación **one-to-many** y una asignación **one-to-one**.

La parte **one-to-many** le da al modelo una supervisión más rica durante el entrenamiento, porque varios candidatos pueden aprender de un mismo objeto. Esto ayuda a que el detector entienda mejor escenas difíciles, especialmente cuando hay muchos objetos cercanos, pequeños o parcialmente ocluidos.

Al mismo tiempo, la parte **one-to-one** obliga al modelo a aprender una predicción más limpia para cada objeto real. En otras palabras, durante la inferencia el modelo ya no depende tanto de producir muchas cajas redundantes para luego eliminarlas, sino que aprende desde antes a generar salidas más precisas y mejor separadas.

En una escena como la de los clones de Naruto, esto es importante porque el reto no es solo detectar personas, sino **distinguir múltiples personas diferentes aunque estén superpuestas**. Gracias a esta asignación dual, YOLOv10 aprende con mayor claridad que hay varias instancias reales ocupando regiones visuales muy parecidas.

Como consecuencia, YOLOv10 reduce la confusión entre personas cercanas y produce predicciones menos redundantes. Así, el problema de que varias cajas compitan entre sí y luego desaparezcan en el postprocesamiento disminuye considerablemente.

En resumen, mientras en YOLOv8 muchas detecciones correctas terminan compitiendo entre sí y el NMS elimina varias de ellas, en YOLOv10 la arquitectura aprende de forma más consistente a representar cada objeto como una instancia distinta. Por eso, en escenarios con oclusión y alta densidad, YOLOv10 ofrece una solución más robusta sin depender de ajustar manualmente el NMS.